### 1. Independent Loading and Initial Cleaning Pipelines (Reddit & Text Datasets)

In [8]:
import pandas as pd
from google.colab import drive
import re
import os

drive.mount('/content/drive')

# Define file paths
text_file_path = '/content/drive/MyDrive/255 Project Data/df_text_cleaned.csv'


# --- Text Dataset Pipeline ---
df_text = pd.read_csv(text_file_path)
# Rename 'status' to 'label'
df_text = df_text.rename(columns={'status': 'label'})
print(f"df_text shape: {df_text.shape}")
print(f"Text labels: {df_text['label'].unique()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
df_text shape: (52681, 3)
Text labels: ['Anxiety' 'Normal' 'Depression' 'Suicidal' 'Stress' 'Bipolar'
 'Personality disorder']


In [9]:
df_text.head()

,statement,label,word_count
0,oh my gosh,Anxiety,3
1,"trouble sleeping, confused mind, restless hear...",Anxiety,10
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety,14
3,I've shifted my focus to something else but I'...,Anxiety,11
4,"I'm restless and restless, it's been a month n...",Anxiety,14


### 2. Stratified 80/20 Split on Dataset

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

print("\n--- Performing Stratified 80/20 Split on the Text Dataset ---")

# Now select the desired columns from the processed df_text
df_text_selected = df_text[['statement', 'label']]

# Perform stratified split on the  data
df_train, df_test = train_test_split(
    df_text_selected,
    test_size=0.2,
    random_state=42,
    stratify= df_text_selected['label'] # Stratify using the label column from the combined data
)

print(f"Train set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")

# Check stratification
print("\nTraining set label distribution:")
print(df_train['label'].value_counts(normalize=True))
print("\nTest set label distribution:")
print(df_test['label'].value_counts(normalize=True))

# Save to parquet files
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_path = os.path.join(output_dir, 'train.parquet')
test_path = os.path.join(output_dir, 'test.parquet')

df_train.to_parquet(train_path, index=False)
df_test.to_parquet(test_path, index=False)
print(f"\nTraining data saved to: {train_path}")
print(f"Testing data saved to: {test_path}")



--- Performing Stratified 80/20 Split on the Text Dataset ---
Train set shape: (42144, 2)
Test set shape: (10537, 2)

Training set label distribution:
label
Normal                  0.310222
Depression              0.292402
Suicidal                0.202188
Anxiety                 0.072917
Bipolar                 0.052700
Stress                  0.049117
Personality disorder    0.020454
Name: proportion, dtype: float64

Test set label distribution:
label
Normal                  0.310240
Depression              0.292398
Suicidal                0.202240
Anxiety                 0.072886
Bipolar                 0.052766
Stress                  0.049065
Personality disorder    0.020404
Name: proportion, dtype: float64

Training data saved to: /content/drive/MyDrive/255 Project Data/train.parquet
Testing data saved to: /content/drive/MyDrive/255 Project Data/test.parquet


### 3. Create undersampling set and Save

In [11]:
print("\n--- Creating Undersampled Training Set ---")

# Get the class distribution of the combined training data
class_counts = df_train['label'].value_counts()
print("Original class distribution in df_combined_train:")
print(class_counts)

# Find the minimum class count
min_class_count = class_counts.min()
print(f"\nSmallest class count: {min_class_count}")

# Create an empty DataFrame to store the undersampled data
df_undersampled_train = pd.DataFrame()

# Undersample each class to the minimum class count
for label in class_counts.index:
    class_df = df_train[df_train['label'] == label]
    # Randomly sample 'min_class_count' entries from the current class
    undersampled_class = class_df.sample(n=min_class_count, random_state=42)
    df_undersampled_train = pd.concat([df_undersampled_train, undersampled_class])

# Shuffle the resulting DataFrame to mix the classes
df_undersampled_train = df_undersampled_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nShape of the undersampled training set: {df_undersampled_train.shape}")
print("\nClass distribution in df_undersampled_train (after undersampling):")
print(df_undersampled_train['label'].value_counts())

# Display the head of the new DataFrame
print("\nHead of df_undersampled_train:")
display(df_undersampled_train.head())

# Save the undersampled DataFrame to parquet
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_undersampled_path = os.path.join(output_dir, 'train_undersampled.parquet')
df_undersampled_train.to_parquet(train_undersampled_path, index=False)
print(f"\nUndersampled training data saved to: {train_undersampled_path}")


--- Creating Undersampled Training Set ---
Original class distribution in df_combined_train:
label
Normal                  13074
Depression              12323
Suicidal                 8521
Anxiety                  3073
Bipolar                  2221
Stress                   2070
Personality disorder      862
Name: count, dtype: int64

Smallest class count: 862

Shape of the undersampled training set: (6034, 2)

Class distribution in df_undersampled_train (after undersampling):
label
Personality disorder    862
Anxiety                 862
Stress                  862
Depression              862
Bipolar                 862
Suicidal                862
Normal                  862
Name: count, dtype: int64

Head of df_undersampled_train:


,statement,label
0,Anyone here feel like they have to be explicit...,Personality disorder
1,"gps, restless panic",Anxiety
2,Survey on Situational Stress and Music (18 and...,Stress
3,All my therapist have added to my trauma I fee...,Depression
4,Disability with Bipolar Disorder I have a gene...,Bipolar



Undersampled training data saved to: /content/drive/MyDrive/255 Project Data/train_undersampled.parquet


### 4. Apply SMOTE to the Training Split and Save

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
import pandas as pd
import os
import numpy as np
from IPython.display import display

print("\n--- Applying SMOTE to the Training Split (Reddit Dataset) ---")
X_train_text = df_train['statement']
y_train = df_train['label']

# Vectorize the text data using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_vectorized = tfidf_vectorizer.fit_transform(X_train_text)
print(f"Original training data shape (vectorized): {X_train_vectorized.shape}")

# Apply SMOTE to the vectorized data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_vectorized, y_train)
print(f"Resampled training data shape (vectorized): {X_resampled.shape}")

# Convert the resampled data back to a dense array and then to a DataFrame
X_resampled_dense = X_resampled.toarray()
feature_names = [f'tfidf_feature_{i}' for i in range(X_resampled_dense.shape[1])]
df_resampled = pd.DataFrame(X_resampled_dense, columns=feature_names)
df_resampled['label'] = y_resampled # Add the resampled labels

print(f"\nFinal df_resampled shape: {df_resampled.shape}")

# Save the resampled DataFrame to parquet
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_smote_path = os.path.join(output_dir, 'train_smote.parquet') # Changed filename to reflect vectorized data
df_resampled.to_parquet(train_smote_path, index=False)
print(f"\nResampled (vectorized) training data saved to: {train_smote_path}")
print(f"Shape of saved parquet: {df_resampled.shape}")
print("\nLabel distribution:")
print(df_resampled['label'].value_counts(normalize=True))

# --- Verification step (loading only a few columns to avoid memory issues) ---
print("\n--- Verifying Parquet file integrity (loading a few columns) ---")
try:
    # Load 'label' and a sample TF-IDF feature column
    columns_to_load = ['label', 'tfidf_feature_0']
    df_reloaded_sample = pd.read_parquet(train_smote_path, columns=columns_to_load)
    print(f"Successfully reloaded selected columns of '{train_smote_path}' with shape: {df_reloaded_sample.shape}")
    print("Sample of reloaded data (first 5 rows): ")
    display(df_reloaded_sample.head())
    print("File appears to be valid and readable.")
except Exception as e:
    print(f"Error reloading '{train_smote_path}': {e}")
    print("This indicates a problem with the file writing process or corruption within the Colab environment.")


--- Applying SMOTE to the Training Split (Reddit Dataset) ---
Original training data shape (vectorized): (42144, 5000)
Resampled training data shape (vectorized): (91518, 5000)

Final df_resampled shape: (91518, 5001)

Resampled (vectorized) training data saved to: /content/drive/MyDrive/255 Project Data/train_smote.parquet
Shape of saved parquet: (91518, 5001)

Label distribution:
label
Depression              0.142857
Normal                  0.142857
Bipolar                 0.142857
Suicidal                0.142857
Stress                  0.142857
Anxiety                 0.142857
Personality disorder    0.142857
Name: proportion, dtype: float64

--- Verifying Parquet file integrity (loading a few columns) ---
Successfully reloaded selected columns of '/content/drive/MyDrive/255 Project Data/train_smote.parquet' with shape: (91518, 2)
Sample of reloaded data (first 5 rows): 


,label,tfidf_feature_0
0,Depression,0.0
1,Normal,0.0
2,Depression,0.0
3,Bipolar,0.0
4,Depression,0.0


File appears to be valid and readable.


### 5. Preprocessing Report: Row Counts at Various Steps (Reddit & Text Datasets)

In [13]:
print("\n--- Preprocessing Report ---\n")

# Reddit Dataset
print("\nText Dataset")
print(f"- Initial rows after loading: {len(df_text)}")
print(f"- Rows in training set after undersampling: {len(df_undersampled_train)}")
print(f"- Rows in training set (before SMOTE): {len(df_train)}")
print(f"- Rows in test set: {len(df_test)}")
print(f"- Rows in training set (after SMOTE): {len(df_resampled)}")



--- Preprocessing Report ---


Text Dataset
- Initial rows after loading: 52681
- Rows in training set after undersampling: 6034
- Rows in training set (before SMOTE): 42144
- Rows in test set: 10537
- Rows in training set (after SMOTE): 91518
